In [0]:
import requests
import pandas as pd
import os
import time
import json

In [0]:
leagues = {
    '61':'league_one',
    '71':'serie_a',
    '39':'premier_league',
    '78':'bundesliga',
    '140':'la_liga'    
}
seasons = ['2021','2022','2023','2024','2025']
teams_url = "https://v3.football.api-sports.io/teams"

api_key = '1b8444b9cb616e2c06743ebd3a22a2ff'
headers = {'x-apisports-key': api_key}

In [0]:
session=requests.Session()

In [0]:
#TEAM DATA
all_teams = []

for league_id, league_name in leagues.items():
    for season in seasons:

        print(f"Fetching {league_name} - {season}")

        while True:
            response = session.get(
                url=teams_url,
                params={
                    "league": league_id,
                    "season": season
                },
                headers=headers,
                timeout=30
            )

            if response.status_code == 429:
                print("Rate limit reached. Waiting 60 seconds...")
                time.sleep(60)
                continue

            response.raise_for_status()
            break

        data = response.json()

        print(f"Retrieved {len(data['response'])} teams")

        for team in data["response"]:
            all_teams.append({
                "league_id": league_id,
                "league_name": league_name,
                "season": season,
                "data": team
            })

        time.sleep(1)


filename = (
    "/Volumes/football_data_platform/raw/source_files/"
    "api_football/team_data/raw_team_data.json"
)

os.makedirs(os.path.dirname(filename), exist_ok=True)

with open(filename, "w", encoding="utf-8") as file:
    json.dump(all_teams, file, indent=2)